# Validate Target Complex Gain calibration

In [ ]:
import os
import h5py
%matplotlib inline

In [ ]:
## required to provide path to notebook utils
import sys

sys.path.insert(0, "../")

In [ ]:
from notebook_utils import identify_max_min_baselineid, plot_phase_vs_time, plot_time_vs_freq_for_phase, plot_time_vs_freq_for_phase_multiple_baselines, plot_phase_vs_freq, plot_data_vs_freq

## Data Generation

We had simulated data using Oskar. The simulation scripts are present in `scripts/ska_low_sim`. (refer [confluence page](https://confluence.skatelescope.org/display/SE/DHR-311%3A+Script+to+simulate+SKA-LOW+visibilities))


### For simulation following configuration is used for calibrator. (further referred to as `test_data_cal.yaml`)

```yaml
scenario: "test_68s_cal_bandpass_lowres"          # Scenario name (used for output folder prefix)

# ===============================
# Global simulation parameters
# ===============================

n_stations: 68                                      # Number of stations
tel_model: "./telescope-models/SKA-Low_AA2_68S_rigid-rotation_model.tm" # Telescope model directory

simulation_start_frequency_hz: 140e6               # Start frequency (Hz)
simulation_end_frequency_hz: 160e6                   # End frequency (Hz)
correlated_channel_bandwidth_hz: 100e3    # Channel width (Hz)

observing_time_mins: 3                              # Observation duration (minutes)
sampling_time_sec: 10.0                   # Dump/integration time (seconds)

fields:
  EoR2:
    Cal1:
      ra_deg: 197.914612
      dec_deg: -22.277973
      scan_id_start: 300
      transit_time: "2000-01-03 22:33:30.000"

# ==================================
# Options for generate_gaintable.py
# ==================================

generate_gaintable:
  output_gaintable: &gen_gaintable "./gaintables/custom_gaintable.h5"

  outlier_config:
    enable: false
    amp_min: 2.0
    amp_max: 5.0
    n_stations_to_corrupt: 20
    n_channels_to_corrupt: 20

  station_offset: true              # Apply per-station amplitude/phase offsets
  time_variant: true                # Apply time-dependent effects

  rfi: false                        # Inject RFI band
  rfi_start_freq_hz: 154.25347222228538e6        # Hz
  rfi_end_freq_hz: 159.8090277778474e6           # Hz

  plot: true                        # Generate diagnostic plots
  plot_output_dir: "./gaintables/generation_plots/"

# ===============================
# Options for run_sim.py
# ===============================

run_sim:
  oskar_sif: "./OSKAR-2.12.2-Python3.sif" # Path to OSKAR Singularity image

  # GLEAM sky model. Optional. Comment to disable.
  gleam_file: "./sky-models/GLEAM_EGC.fits" # GLEAM catalogue FITS file
  field_radius_deg: 10.0            # Radius of field of view (degrees)

  # Corruptions to be applied. All are optional. Comment to disable.
  gaintable: *gen_gaintable           # Gaintable containing bandpass corruptions
  #cable_delay: "./cable_delays/cable_length_error_68s.txt" # Cable delay error file
  #tec_screen: "./tec/tec_scale20e3_rescaled.fits" # Ionospheric TEC screen FITS

  # Imaging parameters using wsclean. Optional. Comment to disable.
  create_dirty_image: false          # Whether to run wsclean imaging
  image_size: 1024                  # Image size (pixels)
  pixel_size: "2arcsec"             # Pixel size (angular units)

  # Extra parameters to pass directly to utils/run_oskar.py
  run_oskar_extra_params: "--use-gpus --double-precision" # Additional command-line parameters for run_oskar.py
```

### Follow steps mentioned in confluence page for data and enviornment setup. Run the following commands to simulate visibilities. 
`python generate_gaintable.py test_data_cal.yaml` \
`python run_sim.py test_data_cal.yaml`

### For the target, the following configuration file is used.

```yaml
scenario: "test_68s_target_all_effects_lowres"          # Scenario name (used for output folder prefix)

# ===============================
# Global simulation parameters
# ===============================

n_stations: 68                                      # Number of stations
tel_model: "./telescope-models/SKA-Low_AA2_68S_rigid-rotation_model.tm" # Telescope model directory

simulation_start_frequency_hz: 140e6               # Start frequency (Hz)
simulation_end_frequency_hz: 160e6                   # End frequency (Hz)
correlated_channel_bandwidth_hz: 100e3    # Channel width (Hz)

observing_time_mins: 10                             # Observation duration (minutes)
sampling_time_sec: 12.0                   # Dump/integration time (seconds)

fields:
  P207+25:         # LoTSS pointing name -- isn't really this pointing though, just kept the same name for consistency with sims -- check coords below for real pointing
    Centre:     # Target coordinates of centre of field
      ra_deg: 129.3395
      dec_deg: -32.8914 #negated the dec
      scan_id_start: 300
      transit_time: "2000-01-03 22:51:30.000" 

# ==================================
# Options for generate_gaintable.py
# ==================================

generate_gaintable:
  output_gaintable: &gen_gaintable "./gaintables/custom_gaintable.h5"

  outlier_config:
    enable: false
    amp_min: 2.0
    amp_max: 5.0
    n_stations_to_corrupt: 20
    n_channels_to_corrupt: 20

  station_offset: true              # Apply per-station amplitude/phase offsets
  time_variant: true                # Apply time-dependent effects

  rfi: false                        # Inject RFI band
  rfi_start_freq_hz: 154.25347222228538e6        # Hz
  rfi_end_freq_hz: 159.8090277778474e6           # Hz

  plot: true                        # Generate diagnostic plots
  plot_output_dir: "./gaintables/generation_plots/"

# ===============================
# Options for run_sim.py
# ===============================

run_sim:
  oskar_sif: "./OSKAR-2.12.2-Python3.sif" # Path to OSKAR Singularity image

  # GLEAM sky model. Optional. Comment to disable.
  lotss_file: "./sky-models/LoTSS_DR2.fits" # LOTSS catalogue FITS file
  field_radius_deg: 10.0            # Radius of field of view (degrees)

  # Corruptions to be applied. All are optional. Comment to disable.
  gaintable: *gen_gaintable           # Gaintable containing bandpass corruptions
  #cable_delay: "./cable_delays/cable_length_error_68s.txt" # Cable delay error file
  #tec_screen: "./tec/tec_scale20e3_rescaled.fits" # Ionospheric TEC screen FITS

  # Imaging parameters using wsclean. Optional. Comment to disable.
  create_dirty_image: false          # Whether to run wsclean imaging
  image_size: 1024                  # Image size (pixels)
  pixel_size: "2arcsec"             # Pixel size (angular units)

  # Extra parameters to pass directly to utils/run_oskar.py
  run_oskar_extra_params: "--use-gpus --double-precision"

```


## Pipeline Setup

First we do bandpass calibration on the simulated calibrator dataset

In [ ]:
cache = "../../cache"
!rm -r {cache}/*
artifacts_prefix_path = "./complex_gain_cal_artefacts"
artifacts_prefix_path = os.path.abspath(artifacts_prefix_path)
!rm -r {artifacts_prefix_path}/*
os.makedirs(artifacts_prefix_path, exist_ok=True)

In [ ]:
data_dir = "/data/abhinav/SKAO/ska-sdp-instrumental-calibration/scripts/ska_low_sim/test_68s_cal_bandpass_lowres-240526_175948"
input_data = f"{data_dir}/visibility.scan-300.ms"
sky_model_path = f"{data_dir}/converted_sky_model.csv"
gleam_file_path = "/home/ska/Work/data/INST/sim/gleamegc.dat"


In [ ]:

## Running the pipeline
!ska-sdp-instrumental-calibration run $input_data \
    --stages load_data predict_vis bandpass_initialisation bandpass_calibration export_gain_table export_visibilities\
    --set parameters.load_data.cache_directory $cache \
    --set parameters.predict_vis.lsm_csv_path  $sky_model_path \
    --set parameters.predict_vis.normalise_at_beam_centre true \
    --set parameters.export_visibilities.data_to_export "vis" \
    --set parameters.export_visibilities.apply_gaintable_to_vis true \
    --dask-scheduler "tcp://127.0.0.1:34567" \
    --output $artifacts_prefix_path \
    --no-unique-output-subdir

## Begin target calibration

We then apply the bandpass gaintables to the simulated target dataset.

In [ ]:
gaintable = f"{artifacts_prefix_path}/UNKNOWN_FIELD_gaintable.h5parm"

In [ ]:
data_dir = "/data/abhinav/SKAO/ska-sdp-instrumental-calibration/scripts/ska_low_sim/test_68s_target_bandpass_lowres-290526_170737"
input_data = f"{data_dir}/visibility.scan-300.ms"
calibrated_data = f"{data_dir}/visibility.scan-300.calibrated.ms"
sky_model_path = f"{data_dir}/converted_sky_model.csv"
gleam_file_path = "/home/ska/Work/data/INST/sim/gleamegc.dat"

Since applybeam is not yet in BPP, we use the follwing DP3 command to apply the beam and gaintable to target data


In [ ]:
DP3_CMD = "singularity exec --bind /data/abhinav/SKAO:/data/abhinav/SKAO /data/abhinav/SKAO/dp3.sif DP3"

if os.path.exists(calibrated_data):
    !rm -r {calibrated_data}

!{DP3_CMD} msin={input_data} msout={calibrated_data} steps=[applybeam,applycal] applycal.type=applycal applycal.parmdb={gaintable} applycal.correction=fulljones applycal.soltab=[amplitude000,phase000] applycal.timeslotsperparmupdate=1

Run complex gain calibration on the calibrated target data

In [ ]:
!rm -r {cache}/*
## Running the pipeline
!ska-sdp-instrumental-target-calibration run $calibrated_data \
    --stages target_load_data predict_vis complex_gain_calibration export_gain_table\
    --set parameters.target_load_data.cache_directory $cache \
    --set parameters.target_load_data.timeslice 9.99 \
    --set parameters.predict_vis.lsm_csv_path  $sky_model_path \
    --set parameters.predict_vis.fov 5.0 \
    --set parameters.predict_vis.flux_limit 0.1 \
    --set parameters.complex_gain_calibration.run_solver_config.niter 500 \
    --set parameters.predict_vis.normalise_at_beam_centre False \
    --set parameters.predict_vis.use_everybeam False \
    --dask-scheduler "tcp://127.0.0.1:34567" \
    --output $artifacts_prefix_path \
    --no-unique-output-subdir

We then apply the finished gaintable to target MS using target bpp

In [ ]:
gaintable = f"{artifacts_prefix_path}/gaintables/complex_gain.gaintable.h5parm"

In [ ]:
data_output_dir = f"{artifacts_prefix_path}/data"
!rm -r {data_output_dir}/*
os.makedirs(data_output_dir, exist_ok=True)
applied_ms = f"{data_output_dir}/visibility.scan-300.calibrated.ms"
beam_corrected_input_data = f"{data_dir}/visibility.scan-300.calibrated.ms"

In [ ]:
target_bpp_config = f"""
steps:
- applycal:
    parmdb: {gaintable}
"""

In [ ]:
from tempfile import NamedTemporaryFile


config_file = NamedTemporaryFile(mode="w", encoding="utf-8", delete=False)
config_file.write(target_bpp_config)
filename = config_file.name
config_file.close()

!ska-sdp-batch-preprocess \
    --config $filename \
    --output-dir $data_output_dir \
    $beam_corrected_input_data

## Imaging

We used wsclean to image the pre and post calibration data and visually inspect the images

In [ ]:
!mkdir -p {artifacts_prefix_path}/images
!wsclean -j 8 -size 6000 6000 -scale 2asec -auto-mask 5 -auto-threshold 3 -niter 1000 -weight briggs 0 -mgain 0.8 -name {artifacts_prefix_path}/images/pre_cal {beam_corrected_input_data}
!wsclean -j 8 -size 6000 6000 -scale 2asec -auto-mask 5 -auto-threshold 3 -niter 1000 -weight briggs 0 -mgain 0.8 -name {artifacts_prefix_path}/images/post_cal {applied_ms}

## Compare Gaintables

We have access to the original gaintable used for corruption. We compare the phases of INST's complex-gain calibration gaintable and the original gaintable 

In [ ]:
from notebook_utils import H5ParmIO
import numpy as np
import matplotlib.pyplot as plt

original_gaintable = '/data/abhinav/SKAO/ska-sdp-instrumental-calibration/scripts/ska_low_sim/gaintables2/custom_gaintable.h5parm'

solset = 'phase000'

original_gains = H5ParmIO.get_values(original_gaintable, solset = solset)

original_gains = np.mean(original_gains, axis=2)

INST_gains = H5ParmIO.get_values(gaintable, solset = solset)

INST_gains = np.squeeze(INST_gains)


In [ ]:
#Rotate the original gaintable w.r.t to the referance antenna=0

refant = 0

ref_phase = original_gains[:, refant, :]

original_gains = original_gains - ref_phase[:,None,:]

In [ ]:
antenna = 2
plt.plot(original_gains[:,antenna,0])
plt.show()

In [ ]:
plt.plot(INST_gains[:,antenna,0])
plt.show()